# adaptationnpu: full pipeline

This notebook installs minimal dependencies, detects available device (NPU/CUDA/CPU), runs a smoke test, performs full training for the original model and the ablated model, and evaluates both on the `test` subset using chunked inference.
Set `MUSDB_ROOT` below once — the notebook will run the full pipeline without further edits.

In [ ]:
# Install Python deps (CPU)
!pip install -r adaptationnpu/requirements.txt

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
try:
    import torch_npu
    print('torch_npu available')
except Exception as e:
    print('torch_npu not available:', e)

In [ ]:
# Configure only MUSDB_ROOT; pipeline uses sensible defaults for training/eval.
import os
MUSDB_ROOT = '/path/to/MUSDB'  # <- set this to your MUSDB folder
OUT_DIR = './out'
DEVICE = 'auto'  # options: 'auto'|'cpu'|'cuda'|'npu'
MODEL = 'ablated'  # default model for smoke test (not required to change)
SR = 16000
# The training script default epochs will be used for full training. Smoke test forces 1 epoch.
# Segment length (seconds) used in training to enforce fixed shapes on NPU
SEGMENT_LENGTH = 3.0
# Inference chunk size (samples) - derived from SEGMENT_LENGTH for parity
CHUNK_SIZE = int(SEGMENT_LENGTH * SR)
# Overlap in seconds for overlap-add during eval (set to 0.0 to disable)
OVERLAP = 0.5
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import sys
import os, argparse
sys.path.append(os.path.abspath("/home/ma-user/work"))

from argparse import Namespace
import train as train_module
import evaluate as eval_module

In [ ]:
# 1) Quick smoke test (1 epoch, short segment) to validate wiring
smoke_out = os.path.join(OUT_DIR, 'smoke')
os.makedirs(smoke_out, exist_ok=True)
#train_module.run_training(musdb_root=MUSDB_ROOT, out_dir=smoke_out, model='ablated', device=DEVICE, segment_length=1.0, sr=SR, batch_size=1, epochs=1, early_stop=5)
!python train.py --musdb-root {MUSDB_ROOT} --out-dir {OUT_DIR}/smoke --model ablated --device npu --segment-length 1.0 --sr 16000 --batch-size 1 --epochs 1 --early-stop 5

In [ ]:
# 2) Full training of original model (uses train.py defaults for epochs)
orig_out = os.path.join(OUT_DIR, 'orig')
os.makedirs(orig_out, exist_ok=True)
train_module.run_training(musdb_root=MUSDB_ROOT, out_dir=orig_out, model='orig', device=DEVICE, segment_length=SEGMENT_LENGTH, sr=SR, batch_size=8, epochs=500, early_stop=30)

In [ ]:

# 3) Full training of ablated model
abl_out = os.path.join(OUT_DIR, 'ablated')
os.makedirs(abl_out, exist_ok=True)
train_module.run_training(musdb_root=MUSDB_ROOT, out_dir=abl_out, model='ablated', device=DEVICE, segment_length=SEGMENT_LENGTH, sr=SR, batch_size=8, epochs=500, early_stop=30)


In [ ]:
import importlib
import evaluate as eval_module
importlib.reload(eval_module) # <--- Clears the Jupyter cache!

from argparse import Namespace
import os

In [ ]:
# 4) Evaluate both models on the test set using chunked inference with overlap-add
chunk = CHUNK_SIZE
overlap = OVERLAP

print('\nEvaluating original model...')
orig_csv = os.path.join(orig_out, 'eval_orig.csv')
args_eval_orig = Namespace(musdb_root=MUSDB_ROOT, model_path=os.path.join(orig_out, 'best.pth'), model='orig', device=DEVICE, chunk_size=chunk, overlap=overlap, out_csv=orig_csv)
eval_module.run_evaluation(args_eval_orig)

print('\nEvaluating ablated model...')
abl_csv = os.path.join(abl_out, 'eval_ablated.csv')
args_eval_abl = Namespace(musdb_root=MUSDB_ROOT, model_path=os.path.join(abl_out, 'best.pth'), model='ablated', device=DEVICE, chunk_size=chunk, overlap=overlap, out_csv=abl_csv)
eval_module.run_evaluation(args_eval_abl)

print('\nPipeline completed successfully.')